# Fashion-MNIST Clothing Classifier

> **Portfolio project:** building and evaluating a convolutional neural network (CNN) for image classification.

## Project at a glance

This project investigates how two CNN design choices affect Fashion-MNIST classification performance:

- **Convolutional padding:** `valid` vs `same`
- **Number of filters:** 16, 32 and 64

The experiments use **stratified 5-fold cross-validation**. The strongest configuration is then retrained on the full training set and evaluated once on the unseen test set.

### Workflow

**Explore → Prepare → Baseline → Controlled experiments → Compare → Select → Retrain → Test → Analyse → Predict**

### What this notebook demonstrates

- Python, NumPy, Pandas and Matplotlib
- TensorFlow / Keras CNN development
- Reusable model-building and evaluation functions
- Stratified cross-validation
- Model comparison and selection
- Confusion-matrix and per-class error analysis
- Model saving and inference

> **Evaluation note:** the 10,000-image test set is kept separate from cross-validation and is used only for the final evaluation.

## 1. Dataset

Fashion-MNIST contains **70,000 grayscale images** across 10 clothing categories. Each image is **28 × 28 pixels**.

| Label | Class |
|---:|---|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

**Dataset split:** 60,000 training images + 10,000 held-out test images.

In [ ]:
# Core libraries
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")

## 2. Explore the data

Before training, confirm the dataset dimensions and inspect representative examples. The sample below shows the raw images before preprocessing.

In [ ]:
# Load the dataset
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print(f"Training images : {train_images.shape}")
print(f"Training labels : {train_labels.shape}")
print(f"Test images     : {test_images.shape}")
print(f"Test labels     : {test_labels.shape}")

In [ ]:
# Visual sample of the training data
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, image, label in zip(axes.ravel(), train_images[:9], train_labels[:9]):
    ax.imshow(image, cmap="gray")
    ax.set_title(class_names[label], fontsize=10)
    ax.axis("off")
fig.suptitle("Sample Fashion-MNIST training images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Class distribution

The classes are evenly represented, so overall accuracy is a useful headline metric alongside the more detailed per-class analysis later in the notebook.

In [ ]:
class_counts = np.bincount(train_labels, minlength=len(class_names))

plt.figure(figsize=(10, 4.5))
plt.bar(class_names, class_counts)
plt.ylabel("Number of training images")
plt.title("Training-set class distribution", fontweight="bold")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

print(f"Images per class: {class_counts.min()} to {class_counts.max()}")

## 3. Prepare the data

Raw pixel values range from 0 to 255. They are normalised to **0–1** and a single grayscale channel is added so the CNN receives images with shape `(28, 28, 1)`.

Labels remain as integer class IDs, allowing sparse categorical cross-entropy without repeatedly creating one-hot encoded labels.

In [ ]:
def prepare_images(images):
    images = images.astype("float32") / 255.0
    return images[..., np.newaxis]

X_train = prepare_images(train_images)
X_test = prepare_images(test_images)

print("Prepared training data:", X_train.shape)
print("Prepared test data:    ", X_test.shape)
print("Pixel range:", X_train.min(), "to", X_train.max())

## 4. CNN architecture

The same overall architecture is used for every experiment so the comparison isolates the variables of interest: **padding** and **filter count**.

**Input (28×28×1) → Conv2D → MaxPooling → Flatten → Dense(100) → Softmax(10)**

Only the convolutional layer's filter count and padding strategy change between experiments.

In [ ]:
def build_cnn(filters=32, padding="valid"):
    """Build the CNN used in the controlled experiments."""
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(
            filters,
            (3, 3),
            padding=padding,
            activation="relu",
            kernel_initializer="he_uniform"
        ),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(100, activation="relu", kernel_initializer="he_uniform"),
        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=0.01,
            momentum=0.9
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

baseline_model = build_cnn(filters=32, padding="valid")
baseline_model.summary()

## 5. Evaluation strategy

Each configuration is evaluated with **5-fold StratifiedKFold cross-validation** on the training set.

For every fold, the model trains on four folds and validates on the remaining fold. Every training example therefore contributes to validation exactly once.

The comparison uses **mean validation accuracy** and its standard deviation. The test set remains untouched until the final model has been selected.

| Setting | Value |
|---|---:|
| Folds | 5 |
| Epochs per fold | 10 |
| Batch size | 32 |
| Random seed | 42 |

In [ ]:
def cross_validate_model(
    X,
    y,
    filters=32,
    padding="valid",
    n_splits=5,
    epochs=10,
    batch_size=32
):
    """Evaluate one CNN configuration with stratified k-fold CV."""
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED
    )

    scores = []
    histories = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        print(f"Fold {fold}/{n_splits}")

        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED + fold)

        model = build_cnn(filters=filters, padding=padding)

        history = model.fit(
            X[train_idx],
            y[train_idx],
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(X[val_idx], y[val_idx]),
            verbose=0
        )

        loss, accuracy = model.evaluate(
            X[val_idx],
            y[val_idx],
            verbose=0
        )

        print(f"  Accuracy: {accuracy * 100:.3f}%")

        scores.append(accuracy)
        histories.append(history)

    return np.array(scores), histories

In [ ]:
def summarise_scores(scores):
    mean_score = scores.mean() * 100
    std_score = scores.std() * 100
    print(f"Mean CV accuracy: {mean_score:.3f}%")
    print(f"Standard deviation: {std_score:.3f} percentage points")
    return mean_score, std_score


def plot_cv_learning_curves(histories, title):
    """Plot the mean training and validation curves across CV folds."""
    train_accuracy = np.mean([h.history["accuracy"] for h in histories], axis=0)
    val_accuracy = np.mean([h.history["val_accuracy"] for h in histories], axis=0)
    train_loss = np.mean([h.history["loss"] for h in histories], axis=0)
    val_loss = np.mean([h.history["val_loss"] for h in histories], axis=0)
    epochs = np.arange(1, len(train_accuracy) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_loss, marker="o", label="Training")
    axes[0].plot(epochs, val_loss, marker="o", label="Validation")
    axes[0].set_title(f"{title}: loss", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    axes[1].plot(epochs, train_accuracy * 100, marker="o", label="Training")
    axes[1].plot(epochs, val_accuracy * 100, marker="o", label="Validation")
    axes[1].set_title(f"{title}: accuracy", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_score_distribution(scores, title):
    plt.figure(figsize=(6, 4))
    plt.boxplot(scores * 100)
    plt.ylabel("Validation accuracy (%)")
    plt.title(title, fontweight="bold")
    plt.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

## 6. Baseline experiment

In [ ]:
baseline_scores, baseline_histories = cross_validate_model(
    X_train,
    train_labels,
    filters=32,
    padding="valid"
)

baseline_mean, baseline_std = summarise_scores(baseline_scores)
plot_cv_learning_curves(baseline_histories, "Baseline CNN")
plot_score_distribution(baseline_scores, "Baseline cross-validation accuracy")

## 7. Padding experiment

In [ ]:
same_scores, same_histories = cross_validate_model(
    X_train,
    train_labels,
    filters=32,
    padding="same"
)

same_mean, same_std = summarise_scores(same_scores)
plot_cv_learning_curves(same_histories, "32 filters + same padding")
plot_score_distribution(same_scores, "32 filters + same padding")

## 8. Increasing the number of filters

In [ ]:
filters64_scores, filters64_histories = cross_validate_model(
    X_train,
    train_labels,
    filters=64,
    padding="same"
)

filters64_mean, filters64_std = summarise_scores(filters64_scores)
plot_cv_learning_curves(filters64_histories, "64 filters + same padding")
plot_score_distribution(filters64_scores, "64 filters + same padding")

## 9. Reducing the number of filters

In [ ]:
filters16_scores, filters16_histories = cross_validate_model(
    X_train,
    train_labels,
    filters=16,
    padding="same"
)

filters16_mean, filters16_std = summarise_scores(filters16_scores)
plot_cv_learning_curves(filters16_histories, "16 filters + same padding")
plot_score_distribution(filters16_scores, "16 filters + same padding")

## 10. Compare the experiments

The table and chart below are generated directly from the cross-validation runs. Sorting by mean accuracy makes the strongest configuration immediately visible.

In [ ]:
results = [
    {"Model": "Baseline", "Filters": 32, "Padding": "valid", "Mean CV Accuracy (%)": baseline_mean, "Std. Dev. (%)": baseline_std},
    {"Model": "Same padding", "Filters": 32, "Padding": "same", "Mean CV Accuracy (%)": same_mean, "Std. Dev. (%)": same_std},
    {"Model": "More filters", "Filters": 64, "Padding": "same", "Mean CV Accuracy (%)": filters64_mean, "Std. Dev. (%)": filters64_std},
    {"Model": "Fewer filters", "Filters": 16, "Padding": "same", "Mean CV Accuracy (%)": filters16_mean, "Std. Dev. (%)": filters16_std}
]

results_df = pd.DataFrame(results).sort_values(
    "Mean CV Accuracy (%)", ascending=False
).reset_index(drop=True)
results_df.insert(0, "Rank", np.arange(1, len(results_df) + 1))

display(results_df.round(3))

plt.figure(figsize=(9, 4.5))
plt.bar(results_df["Model"], results_df["Mean CV Accuracy (%)"])
plt.ylabel("Mean CV accuracy (%)")
plt.title("CNN configuration comparison", fontweight="bold")
plt.xticks(rotation=20, ha="right")
plt.ylim(max(0, results_df["Mean CV Accuracy (%)"].min() - 1.5), 100)
plt.tight_layout()
plt.show()

### Interpreting the comparison

The authoritative result is the controlled comparison produced by this notebook, where every configuration passes through the same 5-fold evaluation function. This is more defensible than selecting a model from a previously recorded figure.

## 11. Select the final configuration

The configuration with the highest **mean cross-validation accuracy** is selected automatically. This keeps model selection tied to the results produced by the current notebook rather than hard-coding a choice from the original coursework.

In [ ]:
best_row = results_df.iloc[0]

FINAL_FILTERS = int(best_row["Filters"])
FINAL_PADDING = best_row["Padding"]

print(f"Selected model: {best_row['Model']}")
print(f"Filters: {FINAL_FILTERS}")
print(f"Padding: {FINAL_PADDING}")
print(f"Mean CV accuracy: {best_row['Mean CV Accuracy (%)']:.3f}%")

## 12. Train the final model

The selected architecture is retrained using **all 60,000 training images**. The test set is still untouched.

The final training curves provide a concise visual check of learning behaviour.

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

final_model = build_cnn(
    filters=FINAL_FILTERS,
    padding=FINAL_PADDING
)

history = final_model.fit(
    X_train,
    train_labels,
    epochs=10,
    batch_size=32,
    verbose=1
)

final_model.save("fashion_mnist_cnn.keras")

print("Model saved as fashion_mnist_cnn.keras")

# Final training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = np.arange(1, len(history.history["accuracy"]) + 1)

axes[0].plot(epochs, history.history["loss"], marker="o")
axes[0].set_title("Final model training loss", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.2)

axes[1].plot(epochs, history.history["accuracy"], marker="o")
axes[1].set_title("Final model training accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


## 13. Final test-set evaluation

This is the first point at which the held-out test set is used. The resulting accuracy is the final estimate of performance on unseen Fashion-MNIST examples.

In [ ]:
test_loss, test_accuracy = final_model.evaluate(
    X_test,
    test_labels,
    verbose=0
)

print(f"Test accuracy: {test_accuracy * 100:.3f}%")
print(f"Test loss: {test_loss:.4f}")

## 14. Error analysis

Accuracy alone can hide which categories are difficult to distinguish. The confusion matrix shows where predictions are correct and where the model confuses one clothing type with another.

In [ ]:
test_probabilities = final_model.predict(X_test, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)

cm = confusion_matrix(test_labels, test_predictions)

fig, ax = plt.subplots(figsize=(9, 9))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Fashion-MNIST Confusion Matrix")
plt.tight_layout()
plt.show()

## 15. Per-class performance

Precision, recall and F1-score provide a more detailed view of performance for each clothing category.

- **Precision:** when the model predicts a class, how often is it correct?
- **Recall:** how many examples of that class does the model find?
- **F1-score:** a balance between precision and recall.

In [ ]:
report_dict = classification_report(
    test_labels,
    test_predictions,
    target_names=class_names,
    output_dict=True
)

metrics_df = pd.DataFrame(report_dict).T.loc[
    class_names, ["precision", "recall", "f1-score"]
]

display(metrics_df.round(3))

ax = metrics_df.plot(kind="bar", figsize=(10, 4.5))
ax.set_title("Per-class precision, recall and F1-score", fontweight="bold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=35)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 16. Example prediction

A single held-out test image is passed through the final model to show the predicted class, known true label and class probabilities.

In [ ]:
sample_index = 0
sample_image = X_test[sample_index]
sample_true_label = test_labels[sample_index]

sample_probabilities = final_model.predict(
    sample_image[np.newaxis, ...], verbose=0
)[0]
sample_prediction = int(np.argmax(sample_probabilities))
confidence = float(np.max(sample_probabilities))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(sample_image.squeeze(), cmap="gray")
axes[0].set_title(
    f"True: {class_names[sample_true_label]}\n"
    f"Predicted: {class_names[sample_prediction]} ({confidence * 100:.1f}%)",
    fontweight="bold"
)
axes[0].axis("off")

axes[1].barh(class_names, sample_probabilities)
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Predicted probability")
axes[1].set_title("Class probabilities", fontweight="bold")
plt.tight_layout()
plt.show()

## 17. Optional: classify an external image

The trained model can also be used as a small inference function. An external image should roughly match the Fashion-MNIST format: grayscale, centred and resized to 28 × 28 pixels.

In [ ]:
from pathlib import Path
from tensorflow.keras.utils import load_img, img_to_array

def predict_external_image(filename, model=final_model):
    path = Path(filename)

    if not path.exists():
        print(f"File not found: {path}")
        print("Add the image to the notebook working directory and run this cell again.")
        return

    image = load_img(
        path,
        color_mode="grayscale",
        target_size=(28, 28)
    )

    array = img_to_array(image).astype("float32") / 255.0
    array = array[np.newaxis, ...]

    probabilities = model.predict(array, verbose=0)[0]
    prediction = int(np.argmax(probabilities))
    confidence = float(np.max(probabilities))

    plt.figure(figsize=(4, 4))
    plt.imshow(array[0].squeeze(), cmap="gray")
    plt.title(
        f"Prediction: {class_names[prediction]}\n"
        f"Confidence: {confidence * 100:.1f}%"
    )
    plt.axis("off")
    plt.show()

    return class_names[prediction], confidence

# Example:
# predict_external_image("sample_image.png")

# Conclusion

This project developed and evaluated a CNN for Fashion-MNIST image classification, focusing on the effect of convolutional **padding** and **filter count**.

The revised workflow uses a consistent 5-fold stratified evaluation process for all configurations, selects the best model from the measured results, and reserves the test set for one final evaluation.

The notebook also goes beyond a single accuracy figure. The experiment comparison, training curves, confusion matrix, per-class metrics and prediction probabilities provide a clearer picture of how the classifier behaves and where it makes mistakes.

### Key takeaway

**dataset → exploration → preprocessing → CNN → controlled experiments → cross-validation → model selection → final training → unseen test evaluation → error analysis → inference**

This makes the notebook a portfolio example of practical machine-learning experimentation rather than simply a model-training exercise.

## References

- Goldsmiths Artificial Intelligence lecture materials.
- Machine Learning Mastery, *How to Develop a CNN From Scratch for Fashion-MNIST Clothing Classification*.
- GeeksforGeeks, *Fashion MNIST with Python Keras and Deep Learning*.
- Fashion-MNIST is loaded through the TensorFlow/Keras dataset API.